# Acoustic features via vocalpy (SAT)

Use `vocalpy.feature.sat.similarity_features` (Sound Analysis Toolbox-style) to compute pitch, amplitude, entropy, frequency-modulation and goodness-of-pitch contours for individual calls, then reduce to per-call scalars.

We'll start with **warble** and **high-freq** calls, comparing singletons vs in-bout members. Same bout definition as `warble_singletons_vs_bouts.ipynb`.

Runs on the cluster only (needs raw WAVs).

## Setup

Cross-platform paths + vocalpy version check. SAT params are tuned for gerbil ultrasound (Nyquist 62.5 kHz at sr = 125 kHz).

In [ ]:
# What this cell does:
#   - Imports everything we'll need.
#   - Picks the right paths automatically (Gily's Mac vs Flatiron cluster).
#   - Imports SAT_PARAMS + the feature helpers from the shared module
#     `vocalization_analysis.acoustic_features` so this code is reusable
#     in other notebooks (e.g. the alarm or HF analyses).

import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import vocalpy
from vocalization_analysis.acoustic_features import (
    SAT_PARAMS, FEATURE_COLUMNS,
    call_wav_path, load_call_slice, load_call_with_context, compute_features,
)

print(f"vocalpy {vocalpy.__version__}")


# --------------------------------------------------------------------------
# Cross-platform paths.
# --------------------------------------------------------------------------
HOST = platform.system()

if HOST == "Darwin":
    DROPBOX              = Path("/Users/gilyginosar/Dropbox (Personal)/Vocalizations_project")
    PARQUET_DIR          = DROPBOX / "Data" / "parquet_cache"
    FIGURES_DIR          = DROPBOX / "Figures" / "vocalpy_features"
    BASE_PROCESSED_AUDIO = None    # raw WAVs aren't synced to Mac
    SAVE_FIGS            = True
elif HOST == "Linux":
    PARQUET_DIR          = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/all_calls/parquet_cache")
    BASE_PROCESSED_AUDIO = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio")
    FIGURES_DIR          = None
    SAVE_FIGS            = False
else:
    raise RuntimeError(f"Unsupported platform: {HOST}")

if SAVE_FIGS:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name, fmt="pdf"):
    """Save a matplotlib figure if we're set up to save, otherwise no-op."""
    if not SAVE_FIGS:
        return
    fig.savefig(FIGURES_DIR / f"{name}.{fmt}", bbox_inches="tight")


DATES_TO_PLOT = ["2025_03", "2025_07", "2025_10", "2026_02"]

print(f"HOST                 = {HOST}")
print(f"PARQUET_DIR          = {PARQUET_DIR}")
print(f"BASE_PROCESSED_AUDIO = {BASE_PROCESSED_AUDIO}")
print(f"SAVE_FIGS            = {SAVE_FIGS}")
print(f"SAT_PARAMS           = {SAT_PARAMS}")
print(f"n features computed  = {len(FEATURE_COLUMNS)}")

## Load + classify

For each call type (warble, high-freq), pool the 4 dates, detect bouts inside each `(date, exp, location)` group, and label every call as `singleton` (bout size 1), `in_bout` (bout size ≥ `MIN_BOUT_SIZE`), or `small_bout` (dropped from comparison). Same bout window as `warble_singletons_vs_bouts.ipynb`.

In [ ]:
# What this cell does:
#   1. Loads all warble + high-freq calls from the 4 date-folder parquets.
#   2. Calls vocalization_analysis.bouts.detect_bouts_for_types to add
#      bout_id / bout_size / bout_position / bout_kind columns.
#
# The thresholds (min_icg_s, max_icg_s, min_bout_size) live in
# vocalization_analysis/bouts.py - tune them THERE, not here, so every
# notebook stays consistent.

from vocalization_analysis.bouts import detect_bouts_for_types, BOUT_THRESHOLDS

CALL_TYPES = ["warble", "high-freq"]

# Show the thresholds we'll use so we can spot mismatches quickly.
print("Bout thresholds (from vocalization_analysis.bouts):")
for ct in CALL_TYPES:
    print(f"  {ct:10s} -> {BOUT_THRESHOLDS[ct]}")


# --------------------------------------------------------------------------
# Step 1: read one parquet per date, keep only call types we care about.
# --------------------------------------------------------------------------
per_date_dfs = []
for date_tag in DATES_TO_PLOT:
    df_one_date = pd.read_parquet(PARQUET_DIR / f"all_calls_{date_tag}.parquet")
    df_one_date = df_one_date[df_one_date["event_type"].isin(CALL_TYPES)]
    per_date_dfs.append(df_one_date)
all_calls = pd.concat(per_date_dfs, ignore_index=True)

print(f"\nLoaded {len(all_calls):,} calls across dates {DATES_TO_PLOT}")


# --------------------------------------------------------------------------
# Step 2: bout detection (run per call type internally).
# --------------------------------------------------------------------------
calls = detect_bouts_for_types(all_calls, CALL_TYPES)


# --------------------------------------------------------------------------
# Sanity prints.
# --------------------------------------------------------------------------
print("\nCalls per (call type, bout_kind):")
print(calls.groupby(["event_type", "bout_kind"]).size().unstack(fill_value=0))

print("\nCalls per (call type, date, bout_kind):")
print(calls.groupby(["event_type", "date_folder", "bout_kind"]).size().unstack(fill_value=0))

## Acoustic helpers

- `load_call_slice` — read just the syllable from the concatenated channel WAV (same helper as `explore_calls_xplatform.ipynb`).
- `compute_features` — wrap a single call's audio into a `vocalpy.Sound`, run SAT `similarity_features` and soundsig `predefined_acoustic_features(ftr_groups=("spectral",))`, then reduce the time-resolved SAT contours to scalars (medians + a few special reductions like "pitch at the loudest frame" and "start/stop pitch from first/last third of frames", mirroring the Intzandt notebook).

Both modules are wrapped in `try/except` so a single bad call returns NaNs instead of crashing the loop.

In [ ]:
# The acoustic helpers all live in vocalization_analysis/acoustic_features.py now.
# Imported in the setup cell above. Brief reminder of what's available:
#
#   SAT_PARAMS            - SAT params dict (sr=125 kHz tuned)
#   FEATURE_COLUMNS       - list of the 20 per-call scalars compute_features returns
#   call_wav_path(base_audio, date_folder, exp, channel, file_num)
#                         - path to the concatenated channel WAV for a given call
#   load_call_slice(base_audio, ..., start_sec, stop_sec)
#                         - read just the syllable's samples; returns (y, sr)
#   load_call_with_context(base_audio, ..., start_sec, stop_sec, window_sec=1.0)
#                         - read ~1 s of audio around a call (for spectrograms)
#                         - returns (y_window, sr, t0_call, t1_call)
#   compute_features(y, sr)
#                         - run vocalpy and return a flat dict keyed by FEATURE_COLUMNS.
#                           NaN on any failure (so a batch loop won't crash).
#
# Path helpers take `base_audio` as their FIRST arg so the module stays platform-pure.
# Pass `BASE_PROCESSED_AUDIO` (defined in the setup cell above) at every call site.

print("Available features:")
for col in FEATURE_COLUMNS:
    print(f"  {col}")

## Sanity check on one call per (call_type, kind)

Pick one example from each of `{warble, high-freq} × {singleton, in_bout}` and run `compute_features`. Print the resulting feature vector and a spectrogram so we can eyeball whether the numbers make sense before running the full extraction loop.

In [ ]:
# What this cell does:
#   1. Pick one example call from each of (warble, high-freq) x (singleton, in_bout).
#   2. Run compute_features() on the call's audio (call slice only).
#   3. Plot a 1-second spectrogram centered on the call (with context),
#      with the extracted scalar features overlaid as horizontal lines.

import soundfile as sf
import librosa

if HOST != "Linux":
    raise RuntimeError("Need raw WAVs - run this cell on the cluster.")


WINDOW_SEC = 1.0     # spectrogram window length around each call
SEED       = 0
PROB_THR   = 0.7     # require classifier confidence >= this

PROB_COLS = {"warble": "meanprob_warble", "high-freq": "meanprob_high-freq"}


# --------------------------------------------------------------------------
# Step 1: pick one example per (call_type, bout_kind).
# --------------------------------------------------------------------------
rng = np.random.default_rng(SEED)
example_rows = []

for call_type in CALL_TYPES:
    prob_col = PROB_COLS[call_type]
    for kind_value in ("singleton", "in_bout"):
        pool = calls[
            (calls["event_type"] == call_type) & (calls["bout_kind"] == kind_value)
        ]
        if prob_col in pool.columns:
            pool = pool[pool[prob_col] >= PROB_THR]
        if pool.empty:
            print(f"  no calls found for {call_type} / {kind_value}")
            continue
        draw_seed = int(rng.integers(0, 1_000_000))
        example_rows.append(pool.sample(1, random_state=draw_seed).iloc[0])


# --------------------------------------------------------------------------
# Step 2: extract features and plot, one example per row.
# --------------------------------------------------------------------------
n_examples = len(example_rows)
fig, axes = plt.subplots(
    nrows=n_examples, ncols=1,
    figsize=(11, 2.8 * n_examples),
    sharex=True,
)
if n_examples == 1:
    axes = [axes]

for ax, row in zip(axes, example_rows):

    # --- Features (computed on the call slice only) ---
    y_call, sr = load_call_slice(
        BASE_PROCESSED_AUDIO,
        row["date_folder"], row["exp"], row["channel"], row["file_num"],
        row["start_time_file_sec"], row["stop_time_file_sec"],
    )
    features = compute_features(y_call, sr)

    header = (
        f"=== {row['event_type']:10s}  {row['bout_kind']:10s}  "
        f"exp {int(row['exp'])} file {int(row['file_num']):03d} "
        f"ch {int(row['channel'])}  @ {row['start_time_file_sec']:.2f}s "
        f"(dur {features['duration_s']*1000:.0f} ms) ==="
    )
    print(f"\n{header}")
    for col in FEATURE_COLUMNS:
        v = features[col]
        if np.isnan(v):
            print(f"  {col:30s} =      nan")
        elif "hz" in col or "freq" in col:
            print(f"  {col:30s} = {v/1000:>8.2f} kHz")
        else:
            print(f"  {col:30s} = {v:>8.3f}")

    # --- Spectrogram of the wider window for context ---
    y_window, sr, t0_call, t1_call = load_call_with_context(
        BASE_PROCESSED_AUDIO,
        row["date_folder"], row["exp"], row["channel"], row["file_num"],
        row["start_time_file_sec"], row["stop_time_file_sec"],
        window_sec=WINDOW_SEC,
    )

    stft = librosa.stft(
        y_window, n_fft=SAT_PARAMS["n_fft"],
        hop_length=SAT_PARAMS["hop_length"], window="hann",
    )
    spectrogram_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

    ax.imshow(
        spectrogram_db,
        origin="lower", aspect="auto",
        extent=[0, len(y_window) / sr * 1000, 0, sr / 2 / 1000],
        cmap="magma", vmin=-80, vmax=0,
    )

    ax.axvline(t0_call * 1000, color="white", lw=1.0, alpha=0.8)
    ax.axvline(t1_call * 1000, color="white", lw=1.0, alpha=0.8)

    overlay_specs = [
        ("peak_freq_hz",   "cyan",    "-",  "peak"),
        ("start_pitch_hz", "lime",    "--", "start"),
        ("stop_pitch_hz",  "orange",  "--", "stop"),
        ("max_pitch_hz",   "magenta", ":",  "max"),
        ("min_pitch_hz",   "yellow",  ":",  "min"),
        ("mean_s_hz",      "white",   ":",  "mean_s"),
    ]
    for col, color, linestyle, label in overlay_specs:
        v = features[col]
        if not np.isnan(v):
            ax.axhline(
                v / 1000,
                color=color, lw=0.9, ls=linestyle,
                label=f"{label} {v/1000:.1f}",
            )

    ax.set_ylim(0, sr / 2 / 1000)
    ax.set_ylabel("kHz")
    ax.set_title(
        f"{row['event_type']}  ·  {row['bout_kind']}  ·  bout_size={int(row['bout_size'])}  "
        f"·  call at {t0_call*1000:.0f}-{t1_call*1000:.0f} ms in window",
        fontsize=10, loc="left",
    )
    ax.legend(loc="upper right", fontsize=7, framealpha=0.6)

axes[-1].set_xlabel("Time within window (ms)")
fig.tight_layout()
plt.show()